## Descripción de variables del dataset

Este dataset contiene información de usuarios de una plataforma de streaming. Cada fila representa un usuario distinto, con datos sobre su edad, plan de suscripción, país, género favorito, tiempo de visualización y relación con soporte técnico.

| Variable | Descripción | Importancia |
|---|---|---|
| `user_id` | Identificador único del usuario | Permite distinguir a cada usuario dentro del dataset |
| `age` | Edad del usuario | Puede influir en los hábitos de consumo y preferencias de contenido |
| `subscription_plan` | Tipo de plan contratado (`Básico`, `Estándar`, etc.) | Es útil para analizar diferencias de consumo según el plan |
| `monthly_watch_time_mins` | Cantidad de minutos vistos por mes | Mide el nivel de actividad o compromiso del usuario con la plataforma |
| `country` | País del usuario | Permite analizar patrones de consumo por ubicación geográfica |
| `favorite_genre` | Género de contenido favorito del usuario | Ayuda a identificar preferencias de contenido |
| `last_login_date` | Fecha del último inicio de sesión | Sirve para detectar usuarios activos o posibles usuarios inactivos |
| `customer_support_tickets` | Cantidad de tickets enviados a soporte técnico | Puede indicar problemas frecuentes, insatisfacción o necesidad de asistencia |

## Observaciones importantes

- `monthly_watch_time_mins` permite analizar qué usuarios consumen más contenido dentro de la plataforma.
- `subscription_plan` puede estar relacionado con el tiempo de visualización mensual.
- `last_login_date` es útil para identificar usuarios activos, inactivos o con riesgo de abandono.
- `customer_support_tickets` puede ayudar a detectar usuarios con problemas recurrentes.
- `country` permite comparar el comportamiento de los usuarios según la región.
- `favorite_genre` puede utilizarse para analizar gustos y recomendar contenido personalizado.
- El dataset puede utilizarse para:
  - análisis exploratorio de datos (EDA),
  - segmentación de usuarios,
  - análisis de comportamiento de consumo,
  - detección de usuarios inactivos,
  - análisis de satisfacción o problemas con soporte,
  - predicción de abandono de usuarios si se agrega una variable objetivo como `churn`.

# 01 - Inspección inicial

Objetivo: comprender la estructura y calidad inicial del dataset sin aplicar transformaciones definitivas.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path("..").resolve()

raw = pd.read_json(ROOT / "data" / "raw" / "streaming_users_dirty.json")
raw.head()


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [2]:
print(f"Filas: {raw.shape[0]} | Columnas: {raw.shape[1]}")
raw.info()


Filas: 8160 | Columnas: 8
<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB


In [3]:
raw.isna().sum().to_frame("nulos")


,nulos
user_id,0
age,0
subscription_plan,0
monthly_watch_time_mins,193
country,0
favorite_genre,240
last_login_date,320
customer_support_tickets,0


In [4]:
pd.DataFrame({"duplicados_exactos": [raw.duplicated().sum()], "user_id_repetidos": [raw.duplicated("user_id").sum()]})


,duplicados_exactos,user_id_repetidos
0,126,160


In [5]:
raw.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,8160.0,NaN,NaN,NaN,13995.433824,2310.81066,10000.0,11987.75,13998.5,15997.25,17999.0
age,8160.0,NaN,NaN,NaN,34.096814,14.511304,-5.0,25.0,33.0,42.0,150.0
subscription_plan,8160,15,Básico,3450,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_watch_time_mins,7967.0,NaN,NaN,NaN,1107.346153,5310.442604,-120.0,489.2,757.4,1045.7,99999.0
country,8160,26,Brasil,1132,NaN,NaN,NaN,NaN,NaN,NaN,NaN
favorite_genre,7920,28,Comedia,1112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_login_date,7840,3062,2026-15-03,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_support_tickets,8160.0,NaN,NaN,NaN,1.80098,11.334969,-1.0,0.0,1.0,1.0,150.0


In [6]:
for col in ["subscription_plan", "country", "favorite_genre"]:
    print(f"\n{col}")
    print(raw[col].value_counts(dropna=False).head(30))



subscription_plan
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

country
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64

favorite_genre
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller       1090
Romance        1090

In [7]:
for col in ["age", "monthly_watch_time_mins", "customer_support_tickets"]:
    q1, q3 = raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, límite superior={q3 + 1.5*iqr:.2f}")


age: Q1=25.00, Q3=42.00, IQR=17.00, límite superior=67.50
monthly_watch_time_mins: Q1=489.20, Q3=1045.70, IQR=556.50, límite superior=1880.45
customer_support_tickets: Q1=0.00, Q3=1.00, IQR=1.00, límite superior=2.50


## Observaciones iniciales

- El dataset tiene usuarios de streaming con variables demográficas, consumo, plan, país, género favorito, fecha de último login y tickets de soporte.
- Hay duplicados exactos y `user_id` repetidos, por lo que la unicidad del registro debe revisarse.
- Las categorías presentan variantes de escritura (`Std`, `Basic`, `Premiun`, países abreviados y géneros en inglés/español).
- Se observan edades imposibles, tiempos de visualización negativos o extremadamente altos, tickets negativos/extremos y fechas inválidas o futuras.
- Estas evidencias orientan la limpieza; todavía no se elimina ni imputa nada en esta etapa.
